# 大數據在鑑別式AI中的應用

## 學習目標

本 Notebook 以「鑑別式 AI」為主軸，示範大數據如何支援分類、預測與異常偵測任務。完成後，你將能夠：

1. 說明鑑別式 AI 與生成式 AI 的差異。
2. 理解結構化、半結構化、非結構化與多模態資料在模型中的角色。
3. 使用 Python 建立簡易分類模型，觀察資料品質、特徵工程與標註策略對結果的影響。
4. 使用 TF-IDF 模擬文字特徵萃取，理解客服分類與內容分類的基本流程。
5. 透過混淆矩陣、F1 分數與 ROC AUC 評估鑑別式模型表現。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並建立可重現的隨機種子。這些套件都是 Google Colab 常見預裝套件。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

import re
from collections import Counter

np.random.seed(42)

print('環境設定完成')
print('可使用套件：numpy, pandas, sklearn, matplotlib, re, collections')


## 核心概念說明

鑑別式 AI 的目標是根據輸入資料判斷類別、風險或機率，例如：

- 金融：判斷交易是否為詐欺、客戶是否可能違約。
- 零售：預測顧客是否流失、商品應歸到哪個類別。
- 客服：判斷訊息屬於物流、帳號、付款或抱怨類型。
- 醫療：根據病歷或影像判斷疾病風險。
- 資安：判斷登入行為是否異常。

在大數據情境下，模型不只依賴資料量，也高度依賴資料品質。常見影響因素包含缺失值、異常值、重複值、標註錯誤、樣本不平衡與資料來源格式不一致。

本章的實作重點是：將不同資料來源轉換成模型可用的特徵，訓練鑑別式模型，並用合適指標評估其效果。


In [ ]:
# ── 示範：結構化資料的信用風險分類 ─────────────────────────
# 這段程式碼建立一組模擬的金融資料，使用邏輯迴歸預測客戶是否違約，並觀察不平衡資料下 accuracy 與 F1 分數的差異。

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

np.random.seed(42)

n = 2000
income = np.random.normal(65000, 18000, n).clip(18000, 150000)
debt_ratio = np.random.beta(2, 5, n)
late_payments = np.random.poisson(0.7, n)
transaction_count = np.random.poisson(35, n)

risk_score = (
    -0.000025 * income
    + 3.0 * debt_ratio
    + 0.45 * late_payments
    - 0.01 * transaction_count
    + np.random.normal(0, 0.7, n)
)
prob_default = 1 / (1 + np.exp(-risk_score))
default = (prob_default > np.quantile(prob_default, 0.82)).astype(int)

df = pd.DataFrame({
    'income': income,
    'debt_ratio': debt_ratio,
    'late_payments': late_payments,
    'transaction_count': transaction_count,
    'default': default
})

X = df[['income', 'debt_ratio', 'late_payments', 'transaction_count']]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('違約比例：', round(y.mean(), 3))
print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print('Precision:', round(precision_score(y_test, y_pred), 3))
print('Recall:', round(recall_score(y_test, y_pred), 3))
print('F1:', round(f1_score(y_test, y_pred), 3))
print('ROC AUC:', round(roc_auc_score(y_test, y_prob), 3))
print('混淆矩陣：')
print(confusion_matrix(y_test, y_pred))


## 輸入資料類型與標註來源

鑑別式 AI 需要將資料整理成「特徵 X」與「標籤 y」。不同資料型態常見處理方式如下：

- 結構化資料：例如收入、交易金額、登入次數，可直接作為表格特徵。
- 半結構化資料：例如 JSON 日誌，需展平欄位後再建模。
- 非結構化資料：例如客服文字、評論與郵件，需透過 TF-IDF、詞頻或其他方法轉成數值。
- 多模態資料：例如交易資料加上文字評論或圖片特徵，實務上常需先分別轉換，再合併成模型輸入。

資料標註也會影響模型。若標註不一致、樣本偏差嚴重，模型可能在測試資料上看似表現良好，但部署到真實場景後失效。


In [ ]:
# ── 示範：半結構化 JSON 日誌展平與異常偵測 ──────────────────
# 這段程式碼模擬登入日誌，將半結構化資料轉成表格，並使用 Isolation Forest 找出疑似異常登入行為。

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

np.random.seed(7)

logs = []
for i in range(500):
    is_risky = np.random.rand() < 0.06
    logs.append({
        'user_id': f'U{i % 80:03d}',
        'event': {
            'hour': np.random.choice(range(24)),
            'failed_attempts': np.random.poisson(0.4 if not is_risky else 4),
            'device_count_7d': np.random.poisson(2 if not is_risky else 8),
            'country_changed': int(is_risky or np.random.rand() < 0.03)
        }
    })

rows = []
for item in logs:
    rows.append({
        'user_id': item['user_id'],
        'hour': item['event']['hour'],
        'failed_attempts': item['event']['failed_attempts'],
        'device_count_7d': item['event']['device_count_7d'],
        'country_changed': item['event']['country_changed']
    })

df = pd.DataFrame(rows)
features = df[['hour', 'failed_attempts', 'device_count_7d', 'country_changed']]

model = IsolationForest(contamination=0.06, random_state=42)
df['anomaly_label'] = model.fit_predict(features)
df['is_anomaly'] = (df['anomaly_label'] == -1).astype(int)

print('資料前 5 筆：')
print(df.head())
print('\n偵測到的異常登入數：', int(df['is_anomaly'].sum()))
print('\n異常樣本範例：')
print(df[df['is_anomaly'] == 1].head(10))


In [ ]:
# ── 示範：客服文字分類的 TF-IDF 輕量模型 ──────────────────
# 這段程式碼用 TF-IDF 取代大型語言模型的文字表示，示範如何將客服訊息轉換成數值特徵並進行分類。

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

texts = [
    '我的包裹還沒收到，物流狀態一直沒有更新',
    '訂單已經出貨但配送進度卡住了',
    '宅配司機沒有聯絡我，請協助查詢物流',
    '商品到貨時間可以再確認一次嗎',
    '我無法登入帳號，系統一直顯示密碼錯誤',
    '忘記密碼，重設連結沒有收到',
    '會員帳號被鎖定，請幫我解除',
    '手機驗證碼一直失敗，不能登入',
    '信用卡付款失敗，但銀行已經扣款',
    '發票資訊填錯了，付款資料要修改',
    '請問可以更換付款方式嗎',
    '結帳時顯示交易逾時，訂單沒有成立',
    '收到的商品有瑕疵，我要申請退貨',
    '客服回覆太慢，問題一直沒有解決',
    '商品和網頁描述不符，我想提出抱怨',
    '包裝破損而且內容物缺少，請處理',
    '物流顯示已送達，但我沒有收到包裹',
    '配送地址想要修改，包裹還沒出貨',
    '帳號資料想更新，但頁面無法儲存',
    '付款成功後沒有收到訂單確認信',
    '我對售後服務不滿意，希望有人回覆',
    '退貨流程太複雜，請提供協助'
]

labels = [
    '物流', '物流', '物流', '物流',
    '帳號', '帳號', '帳號', '帳號',
    '付款', '付款', '付款', '付款',
    '抱怨', '抱怨', '抱怨', '抱怨',
    '物流', '物流', '帳號', '付款', '抱怨', '抱怨'
]

text_df = pd.DataFrame({
    'message': texts,
    'label': labels
})

X_train, X_test, y_train, y_test = train_test_split(
    text_df['message'],
    text_df['label'],
    test_size=0.3,
    random_state=42,
    stratify=text_df['label']
)

text_model = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))),
    ('clf', LogisticRegression(max_iter=1000))
])

text_model.fit(X_train, y_train)
y_pred = text_model.predict(X_test)

print('測試資料分類報告：')
print(classification_report(y_test, y_pred, zero_division=0))

new_messages = [
    '包裹配送進度沒有更新',
    '重設密碼的驗證信沒有收到',
    '信用卡刷卡失敗但帳戶被扣款',
    '商品破損，我想客訴'
]

predictions = text_model.predict(new_messages)
print('新訊息預測：')
for message, label in zip(new_messages, predictions):
    print(f'{message} -> {label}')
